# MAGAT — run everything from here

One cell per stage. Every cell is independently re-runnable: setup cells skip themselves once their
outputs exist, and every stage ends with a `bundle(...)` cell. Run output lives under `~/magat_runs`
(or `MAGAT_DATA`), **outside the clone**.

## Parameters and helpers

Run this first, always. It finds the repository on its own (a Jupyter kernel's working directory is
the notebook's folder, not where the server was launched) and puts the data root outside the clone.

In [ ]:
import os, sys, subprocess, pathlib, datetime, zipfile, glob

MAGAT_ROOT = ""     # leave empty: auto-detected. Set an absolute path only if the banner says so.

def _find_root():
    if MAGAT_ROOT:
        return pathlib.Path(MAGAT_ROOT).resolve()
    here = pathlib.Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / "magat.py").is_file() and (base / "core" / "src").is_dir():
            return base
        for child in base.glob("*"):
            if child.is_dir() and (child / "magat.py").is_file() and (child / "core" / "src").is_dir():
                return child
    raise RuntimeError(
        f"Cannot find the MAGAT repository at or above {here}. A Jupyter kernel's working directory "
        "is the notebook's own folder (and it keeps the OLD folder if the file was moved without a "
        "kernel restart). Restart the kernel, or set MAGAT_ROOT at the top of this cell.")

ROOT = _find_root()
os.chdir(ROOT)

# Everything a run produces lives OUTSIDE the clone, so a merge and a re-clone never touch it.
DATA = pathlib.Path(os.environ.get("MAGAT_DATA") or (pathlib.Path.home() / "magat_runs")).resolve()
os.environ["MAGAT_DATA"] = str(DATA)
LOGS = DATA / "logs"
BUNDLES = DATA / "bundles"
for _d in (LOGS, BUNDLES):
    _d.mkdir(parents=True, exist_ok=True)
assert ROOT not in DATA.parents and DATA != ROOT, f"data root {DATA} must be outside the clone {ROOT}"

GAME = "tribes"
PY = sys.executable
PUBLISHED_PORTFOLIO = ROOT / "games" / GAME / "portfolio" / "v1"
PUBLISHED_WEIGHTS = ROOT / "games" / GAME / "calibration" / "r7615" / "compute_weights.properties"
# Workers for parallel stages. Half the logical CPUs (SMT siblings add heat, not throughput) less two
# for the machine; MAGAT_WORKERS overrides. The machine profile (milestone 3) replaces this guess.
WORKERS = int(os.environ.get("MAGAT_WORKERS") or max(1, (os.cpu_count() or 2) // 2 - 2))

def magat(*args, check=True):
    """Run one launcher command; the launcher tees its own output to <DATA>/logs/<name>.log."""
    cmd = [PY, str(ROOT / "magat.py"), *map(str, args)]
    print("$ " + " ".join(cmd[2:]), flush=True)
    rc = subprocess.call(cmd)
    if check and rc != 0:
        raise RuntimeError(f"exit {rc}: magat {' '.join(map(str, args))}")
    return rc

def latest_log(name):
    txt = (LOGS / f"{name}.log").read_text(encoding="utf-8", errors="replace")
    return txt.rsplit("\n>>>>> ", 1)[-1]

def assert_in_log(name, needle):
    assert needle in latest_log(name), f"'{needle}' not found in the latest {name} run - read {LOGS / (name + '.log')}"
    print(f"[gate] '{needle}' confirmed in {name}")

def summary(results, until="## Budget multiplier"):
    """Print an analysis's SUMMARY.md up to a section (the provenance, the side by side and the headline)."""
    text = (pathlib.Path(results) / "SUMMARY.md").read_text(encoding="utf-8")
    cut = text.find(until)
    print(text if cut < 0 else text[:cut] + f"... the rest in {pathlib.Path(results) / 'SUMMARY.md'}")

def bundle(name, items, exclude=("seqlog_", "mutlog_", "_running.lock")):
    """Zip the pre-specified files of a stage into <DATA>/bundles/<name>.zip."""
    out = BUNDLES / f"{name}.zip"
    if out.exists():
        out.unlink()
    files = []
    for it in items:
        p = pathlib.Path(it)
        if not p.is_absolute():
            p = DATA / p
        if p.is_dir():
            files += [q for q in p.rglob("*") if q.is_file()]
        elif p.exists():
            files.append(p)
        else:
            files += [pathlib.Path(h) for h in glob.glob(str(p)) if pathlib.Path(h).is_file()]
    files = sorted({f for f in files if not any(x in f.name for x in exclude)})
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for f in files:
            z.write(f, str(f.relative_to(DATA)) if DATA in f.parents else f.name)
    print(f"{len(files)} files -> {out}")
    return out

print("repository :", ROOT)
print("data root  :", DATA, "(outside the clone)")
print("python     :", PY)
print("workers    :", WORKERS)

## Build, tests, layering

In [ ]:
magat("build", "--tests")
magat("test")
magat("layering")

## Portfolio

Build this machine's own copy of the portfolio from the game's scenario sources, freeze its split,
and verify it against the published one: the same identity, every manifest row equal, and the same
plans from both sets of scenario files. A rebuild that matches is the evidence that positions, and
therefore every published result, reproduce here. Skips itself once the portfolio is built and frozen;
the journal makes an interrupted build resume where it stopped.

In [ ]:
PORTFOLIO = DATA / "portfolio" / GAME
if (PORTFOLIO / "_split_dev.txt").exists() and (PORTFOLIO / "_split_holdout.txt").exists():
    print(f"[skip] {PORTFOLIO} is built and frozen")
else:
    magat("portfolio", "build", "--game", GAME, "--out", PORTFOLIO, "--threads", WORKERS)
    magat("portfolio", "freeze", "--portfolio", PORTFOLIO)
print((PORTFOLIO / "split_report.txt").read_text(encoding="utf-8"))
magat("portfolio", "verify", "--game", GAME, "--portfolio", PORTFOLIO, "--published", PUBLISHED_PORTFOLIO, "--plan", 4)
assert_in_log("portfolio-verify", "VERIFY: PASS")
bundle("portfolio", [PORTFOLIO / f for f in ("portfolio.csv", "_split_dev.txt", "_split_holdout.txt",
                                             "identity.txt", "split_report.txt")]
       + [LOGS / "portfolio-build.log", LOGS / "portfolio-freeze.log", LOGS / "portfolio-verify.log"])

## Machine, threads, calibration

The machine banner first: physical cores, the worker ceiling and why. Then the thread sweep, which
picks the knee of sustained throughput per budget level (results do not depend on it, only wall-clock
does). Then this machine's own currency, idle and single-threaded, and the drift check against the
published one. Each skips itself once its output exists.

In [ ]:
magat("machine")

In [ ]:
CALIBRATION = DATA / "calibration" / GAME
if (CALIBRATION / "threads.json").exists():
    print(f"[skip] {CALIBRATION / 'threads.json'} exists")
else:
    magat("threads", "--game", GAME, "--portfolio", PORTFOLIO, "--out", CALIBRATION)
print((CALIBRATION / "threads.json").read_text(encoding="utf-8"))
bundle("threads", [CALIBRATION / "threads.json", CALIBRATION / "thread_sweep.csv", LOGS / "threads.log"])

In [ ]:
# This machine's own currency, on this machine's own portfolio. Idle and single-threaded: stop
# everything else first, and plug a laptop in (on battery it runs at a fraction of its speed).
# Version 2 (CURRENCY_VERSION = 2) prices every operation an arm declares, its own work included;
# version 1 prices the forward model only, as the published R7615 currency does. The estimate runs first:
# one repetition of the reference cell per level, a few minutes, then what the whole session will take.
# Extending the currency later with a new arm is `--arms <new>`, which merges in the file's own version.
CURRENCY_VERSION = 2
if (CALIBRATION / "compute_weights.properties").exists():
    print(f"[skip] {CALIBRATION / 'compute_weights.properties'} exists")
else:
    magat("calibrate", "--estimate", "--game", GAME, "--portfolio", PORTFOLIO, "--format", CURRENCY_VERSION)
    magat("calibrate", "--game", GAME, "--portfolio", PORTFOLIO, "--out", CALIBRATION, "--format", CURRENCY_VERSION)
    assert_in_log("calibrate", "CALIBRATION: PASS")
WEIGHTS = CALIBRATION / "compute_weights.properties"     # the currency every run below spends
magat("weights", "--weights", WEIGHTS)
bundle("calibration", [CALIBRATION / "compute_weights.properties", CALIBRATION / "compute_profiles.csv",
                       CALIBRATION / "compute_operations.csv", LOGS / "calibrate.log"])

In [ ]:
# Does this machine still spend the published currency? Re-measures the reference arms against the
# R7615 file and writes nothing. Run it before quoting anything beside published results.
magat("calibrate", "--check", "--game", GAME, "--portfolio", PORTFOLIO, "--arms", "emcts,emcts-comma",
      "--weights", PUBLISHED_WEIGHTS, check=False)
bundle("drift", [LOGS / "calibrate-check.log"])

In [ ]:
# Gate A3' (off by default). The thesis resolved it: the idle currency is the ruler and wall-clock
# (W7) is co-primary, so this is a diagnostic, not a step. It writes compute_weights_load<W>.properties
# beside the idle file, never over it, and prints the idle-versus-loaded ratios.
RUN_UNDER_LOAD = False
if RUN_UNDER_LOAD:
    magat("calibrate", "--game", GAME, "--portfolio", PORTFOLIO, "--out", CALIBRATION,
          "--arms", "emcts,emcts-comma", "--under-load", WORKERS)

## Gates and probes

The contract laws on published positions; the equivalence gate, where every ported arm must reproduce
its thesis-tree oracle field for field, mechanism counters included (all of them: about ten minutes);
the structure check; the replay of published R7615 holdout turns on this machine's own portfolio; the
support probe for the biased arms. Nothing downstream runs otherwise.

In [ ]:
magat("check", "contract", "--game", GAME, "--scenarios", 8)
assert_in_log("check-contract", "CONTRACT: PASS")
magat("check", "equivalence", "--game", GAME, "--scenarios", 8, "--visits", 2, "--budgets", "8000,40000")
assert_in_log("check-equivalence", "EQUIVALENCE: PASS")
magat("check", "structure", "--game", GAME, "--spec", ROOT / "experiments" / GAME / "structure.json")
assert_in_log("check-structure", "STRUCTURE: PASS")
# The audit: each arm's declared operations and forward model account for its turn (fails above 25% unaccounted).
magat("check", "overhead", "--audit", "--game", GAME, "--portfolio", PORTFOLIO, "--arms", "@reference", "--budget", 40000)
assert_in_log("check-overhead", "OVERHEAD AUDIT: PASS")
magat("check", "replay", "--game", GAME, "--leaf", ROOT / "games" / GAME / "results" / "replay" / "Comma_Off" / "80000cu",
      "--portfolio", PORTFOLIO, "--weights", PUBLISHED_WEIGHTS, "--turns", 20)
assert_in_log("check-replay", "REPLAY: PASS")
magat("probe", "support", "--game", GAME, "--arm", "comma-bias-e25", "--scenarios", 20, "--weights", WEIGHTS,
      "--out", CALIBRATION, check=False)      # a failing level is a reporting level, not an error
bundle("gates", [LOGS / f"{n}.log" for n in ("check-contract", "check-equivalence", "check-structure", "check-replay",
                                             "check-overhead", "probe-support", "test")] + [CALIBRATION / "support_probe_comma-bias-e25.csv"])

## The worked result, re-analysed

The thesis's AB-EMCTS line, imported into MAGAT's run layout: every measure of the brief, in seconds,
from the published rows. A newcomer's first look at what a MAGAT report says.

In [ ]:
# The published worked result (the thesis's AB-EMCTS line, imported), re-analysed here in seconds. It must
# reproduce games/tribes/results/ab-emcts/report; docs/results/tribes-ab-emcts.md sets it beside the
# thesis retrospective, number by number.
WORKED = DATA / GAME / "results" / "ab-emcts-published"
magat("analyze", "--run", ROOT / "games" / GAME / "results" / "ab-emcts", "--out", WORKED)
assert_in_log("analyze", "ANALYZE: OK")
summary(WORKED, until="## Headline (dev)")
bundle("worked-result", [WORKED / "SUMMARY.md", WORKED / "tables", LOGS / "analyze.log"])

## Experiment: ab-emcts-dev

The worked result's development stages (the thesis line's stages D and E, R7615, 2026-08-29 to 2026-08-31) on the 200 development scenarios. The published run is imported in games/tribes/results/ab-emcts/dev; running this re-measures it on this machine.

Split `dev`; baseline `emcts-comma`; levels (budget x visits) 80,000 x 8, 40,000 x 8, 400,000 x 6; arms emcts, emcts-bias-e25, emcts-comma, comma-bias-e75, comma-bias-e50, comma-bias-e25, comma-bias-e10, comma-bias-coarse-e25, comma-bias-fine-e25, comma-bias-norem-e25, comma-bias-perburn-e25, comma-short15, comma-short15-bias-e25, comma-short5-bias-e25, comma-rand15, comma-short15-coarse, comma-z0-coarse, comma-z0-mid, comma-z0-fine, comma-z0-random
(370 turns per scenario of the split). Re-running the cell resumes: finished leaves are skipped,
a partial leaf continues where it stopped.

In [ ]:
SPEC = ROOT / "experiments" / GAME / "ab-emcts-dev.json"
RUN = DATA / GAME / "runs" / "ab-emcts-dev"
RESULTS = DATA / GAME / "results" / "ab-emcts-dev"
magat("run", "--experiment", SPEC, "--split", "dev", "--portfolio", PORTFOLIO, "--weights", WEIGHTS, "--out", RUN)
assert_in_log("run-dev", "RUN: COMPLETE")
magat("analyze", "--run", RUN / "dev", "--out", RESULTS)
assert_in_log("analyze", "ANALYZE: OK")
summary(RESULTS)
bundle("run-ab-emcts-dev", [RUN / "dev", RESULTS / "SUMMARY.md", RESULTS / "tables"])

## Experiment: ab-emcts-holdout

The worked result's holdout read (the thesis line's stage F, R7615, 2026-09-02): the champion, its attribution controls and the repair-granularity sweep on the 200 held-out scenarios. The published run is imported in games/tribes/results/ab-emcts/holdout; running this re-reads the holdout on this machine.

Split `holdout`; baseline `emcts-comma`; levels (budget x visits) 80,000 x 8, 400,000 x 6; arms emcts-comma, comma-z0-coarse, comma-z0-random, comma-rand15, comma-short15-coarse, comma-short15, comma-z0-mid, comma-z0-fine
(112 turns per scenario of the split). Re-running the cell resumes: finished leaves are skipped,
a partial leaf continues where it stopped.

In [ ]:
SPEC = ROOT / "experiments" / GAME / "ab-emcts-holdout.json"
RUN = DATA / GAME / "runs" / "ab-emcts-holdout"
RESULTS = DATA / GAME / "results" / "ab-emcts-holdout"
FROZEN = SPEC.parent / "ab-emcts-frozen.txt"
# 1. Drift: does this machine still spend the currency the development half was priced in? Writes nothing.
magat("calibrate", "--check", "--game", GAME, "--portfolio", PORTFOLIO, "--arms", "emcts-comma", "--weights", WEIGHTS, check=False)
# 2. The holdout is read once, and only for choices written down before it is read.
assert FROZEN.is_file(), f"no frozen choices at {FROZEN}: write them, from development only, before reading the holdout"
print(FROZEN.read_text(encoding="utf-8"))
magat("run", "--experiment", SPEC, "--split", "holdout", "--portfolio", PORTFOLIO, "--weights", WEIGHTS, "--frozen", FROZEN, "--out", RUN)
assert_in_log("run-holdout", "RUN: COMPLETE")
DEV = DATA / GAME / "runs" / "ab-emcts-dev" / "dev"
magat("analyze", "--run", DEV, "--holdout", RUN / "holdout", "--out", RESULTS)
assert_in_log("analyze", "ANALYZE: OK")
summary(RESULTS)
bundle("run-ab-emcts-holdout", [RUN / "holdout", RESULTS / "SUMMARY.md", RESULTS / "tables", LOGS / "calibrate-check.log"])

## Experiment: reference-ladder

The registered reference arms at the published budget ladder: the field every new arm is compared against. The worked result's own specs arrive with its arms (milestone 5).

Split `dev`; baseline `emcts`; levels (budget x visits) 80,000 x 8, 40,000 x 8, 400,000 x 6; arms emcts, emcts-comma, random
(66 turns per scenario of the split). Re-running the cell resumes: finished leaves are skipped,
a partial leaf continues where it stopped.

In [ ]:
SPEC = ROOT / "experiments" / GAME / "reference-ladder.json"
RUN = DATA / GAME / "runs" / "reference-ladder"
RESULTS = DATA / GAME / "results" / "reference-ladder"
magat("run", "--experiment", SPEC, "--split", "dev", "--portfolio", PORTFOLIO, "--weights", WEIGHTS, "--out", RUN)
assert_in_log("run-dev", "RUN: COMPLETE")
magat("analyze", "--run", RUN / "dev", "--out", RESULTS)
assert_in_log("analyze", "ANALYZE: OK")
summary(RESULTS)
bundle("run-reference-ladder", [RUN / "dev", RESULTS / "SUMMARY.md", RESULTS / "tables"])

## Shakedown

Every moving part at toy size, minutes, exits non-zero on any failure. Run before every pull request.

In [ ]:
magat("shakedown")
bundle("shakedown", [LOGS / "shakedown.log"])

---
*The inventory cell (what to keep, with a tick per file) arrives with milestone 7 (see `MAGAT/DESIGN.md` §16).*